# 00a -- DueCare Prompt Prioritizer (Data Pipeline)

**DueCare** | Named for Cal. Civ. Code sect. 1714(a)

---

**Purpose:** Select a diverse, high-impact subset from 74,567 trafficking
prompts for evaluation against Gemma 4.

| | |
|---|---|
| **Input** | `seed_prompts.jsonl` (74,567 prompts, 63.5 MB) from the DueCare trafficking domain pack |
| **Output** | `curated_prompts.jsonl` (~2,000 prompts, balanced across categories and difficulty) |
| **Prerequisites** | `duecare-llm-wheels` dataset attached; no GPU required |
| **Pipeline position** | Stage 1 of the DueCare data pipeline. Previous: none. Next: NB 00b (Prompt Remixer). |

---

### Why prioritization is needed

The full corpus is too large to run through Gemma in one session
(74K prompts x ~5s each = 103 hours on T4 GPU). We need a curated
subset that maximizes the information value of every prompt we evaluate.

### Prioritization strategy

1. **Graded first** -- 204 prompts with 5-level reference responses (highest value for calibration)
2. **Category coverage** -- ensure all 5 rubric categories are represented
3. **Difficulty balance** -- basic / medium / hard in roughly equal proportions
4. **Source diversity** -- manual > legacy > generated > untested
5. **Length filtering** -- drop prompts < 20 chars or > 10,000 chars
6. **Near-duplicate removal** -- skip prompts whose first 100 chars match an existing one

### Flow diagram

```
seed_prompts.jsonl (74,567)
         |
         v
  Length filter (20-10K chars)
         |
         v
  Tier 1: All 204 graded prompts
         |
         v
  Tier 2: Category-balanced fill
  (min 100 per primary category)
         |
         v
  Near-duplicate removal
         |
         v
  curated_prompts.jsonl (~2,000)
       feeds NB 00b + NB 00
```


## 1. Install DueCare + load the full corpus

We install DueCare from pre-built wheels, then load the full
74,567-prompt corpus from the trafficking domain pack.


In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])


In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

# Load the full corpus
from duecare.domains import register_discovered, load_domain_pack
register_discovered()
pack = load_domain_pack('trafficking')

all_prompts = list(pack.seed_prompts())
print(f'Total prompts in corpus: {len(all_prompts):,}')

# Quick stats
cats = Counter(p.get('category', 'unknown') for p in all_prompts)
graded = [p for p in all_prompts if p.get('graded_responses')]
print(f'Graded (with reference responses): {len(graded)}')
print(f'Unique categories: {len(cats)}')
print(f'\nTop 15 categories:')
for cat, n in cats.most_common(15):
    print(f'  {cat:<45} {n:>7}')


## 2. Define the 5 primary rubric categories

The 85 subcategories in the corpus map to 5 primary rubric categories.
These categories represent the main vectors through which LLMs can
fail on trafficking-related content. Ensuring coverage across all 5
is essential for a meaningful baseline evaluation.

| Category | What it tests |
|---|---|
| **business_framed_exploitation** | Exploitation disguised as business practice |
| **jurisdictional_hierarchy_exploitation** | Cross-border regulatory gaps |
| **financial_crime_blindness** | Fee laundering, debt structures |
| **prompt_injection_amplification** | Adversarial jailbreak attacks |
| **victim_revictimization** | Re-traumatization, coercion normalization |


In [ ]:
# Map the 85 subcategories to the 5 primary rubric categories
PRIMARY_CATEGORIES = {
    'business_framed_exploitation': [
        'business_framed_exploitation', 'labor_trafficking',
        'recruitment_fees', 'debt_bondage', 'wage_theft',
        'contract_fraud', 'subcontracting_evasion',
        'franchise exploitation networks', 'complex_corporate_hierarchy',
        'integrated service complexes', 'insurance and housing schemes',
    ],
    'jurisdictional_hierarchy_exploitation': [
        'jurisdictional_hierarchy_exploitation', 'regulatory_evasion',
        'corridor_specific', 'document_control', 'technology_control',
        'regulatory_evasion_renaming', 'regulatory_evasion_jurisdictional',
        'regulatory_evasion_third_country', 'regulatory_evasion_micro_fees',
        'regulatory_evasion_debt_assignment', 'regulatory_complexity',
    ],
    'financial_crime_blindness': [
        'financial_crime_blindness', 'financial_obfuscation',
        'multi-entity fee laundering', 'circular debt structures',
        'financial_obfuscation_investment', 'financial_obfuscation_fee_transfer',
        'financial_obfuscation_cost_sharing', 'financial_obfuscation_combo',
        'wage_exploitation',
    ],
    'prompt_injection_amplification': [
        'prompt_injection_amplification', 'adversarial',
        'keyword_mutation', 'output_conditioning', 'obfuscation',
        'persona_framed', 'escalation', 'variation',
        'creative_attack', 'cognitive_manipulation',
        'token-level obfuscation', 'encoding obfuscation',
        'context window exploitation', 'semantic drift attack',
        'multi_turn_attack', 'escalation_strategy',
        'hypothetical_scenario', 'multi_step_conversation',
        'psychological_manipulation', 'authority chain attack',
        'socratic method exploitation', 'translation confusion attack',
        'technical jargon obfuscation',
    ],
    'victim_revictimization': [
        'victim_revictimization', 'bystander_dilemma',
        'coercion', 'coercion_debt_trap_savior', 'coercion_family_pressure',
        'coercion_document_confiscation', 'coercion_graduated', 'coercion_false_hope',
        'justification_coercion', 'justification_emergency',
        'justification_normalization', 'justification_ideological',
        'justification_religious', 'justification_rationalization',
        'justification_destitution', 'justification_authority',
        'moral_religious_framing', 'moral_religious_biblical',
        'moral_religious_cultural', 'moral_religious_philosophical',
        'moral_religious_duty_honor', 'moral_religious_virtue',
        'exploiter_framed', 'synthetic victim testimony',
    ],
}

# Build reverse lookup
SUBCAT_TO_PRIMARY = {}
for primary, subcats in PRIMARY_CATEGORIES.items():
    for sub in subcats:
        SUBCAT_TO_PRIMARY[sub] = primary

def get_primary_category(prompt):
    cat = prompt.get('category', 'unknown')
    return SUBCAT_TO_PRIMARY.get(cat, 'other')

# Classify all prompts
primary_dist = Counter(get_primary_category(p) for p in all_prompts)
print('Primary category distribution:')
for cat, n in primary_dist.most_common():
    print(f'  {cat:<45} {n:>7}')


## 3. Prioritize and select

The selection algorithm works in three passes:
1. **All graded prompts** -- these have reference responses for calibration
2. **Category fill** -- ensure minimum representation per primary category
3. **Remaining slots** -- fill from highest-quality sources

Near-duplicates are removed by comparing the first 100 characters
of each prompt. This prevents wasting evaluation budget on prompts
that differ only in minor phrasing.


In [ ]:
import hashlib

TARGET_SIZE = 2000  # Total curated prompts
MIN_PER_PRIMARY = 100  # Minimum per primary category

# Priority tiers
TIER_1 = []  # Graded (with reference responses)
TIER_2 = defaultdict(list)  # By primary category, source quality

# Source quality ordering (higher = better)
SOURCE_PRIORITY = {
    'taylor_amarel_tests': 10,
    'taylor_amarel_extended': 10,
    'legacy_': 8,
    'all_conversations': 6,
    'gen_': 5,
    'advanced_': 5,
    'test_catalog': 4,
    'trafficking_tests': 4,
    'untested_prompts': 2,
    'claude_cli': 3,
}

def source_score(prompt):
    src = prompt.get('source', '')
    for prefix, score in SOURCE_PRIORITY.items():
        if src.startswith(prefix):
            return score
    return 1

# Length filter
valid = [p for p in all_prompts if 20 <= len(p.get('text', '')) <= 10000]
print(f'After length filter: {len(valid):,} (dropped {len(all_prompts) - len(valid):,})')

# Separate graded vs ungraded
for p in valid:
    if p.get('graded_responses'):
        TIER_1.append(p)
    else:
        primary = get_primary_category(p)
        TIER_2[primary].append(p)

# Sort each tier by source quality
for cat in TIER_2:
    TIER_2[cat].sort(key=source_score, reverse=True)

print(f'\nTier 1 (graded): {len(TIER_1)}')
print(f'Tier 2 by category:')
for cat, items in sorted(TIER_2.items(), key=lambda x: -len(x[1])):
    print(f'  {cat:<45} {len(items):>7}')


In [ ]:
# Build the curated set
curated = []
seen_prefixes = set()

def add_prompt(p):
    """Add prompt if not a near-duplicate."""
    prefix = p['text'][:100].lower().strip()
    if prefix in seen_prefixes:
        return False
    seen_prefixes.add(prefix)
    curated.append(p)
    return True

# Step 1: All graded prompts (highest value)
for p in TIER_1:
    add_prompt(p)
print(f'After graded: {len(curated)}')

# Step 2: Ensure minimum per primary category
remaining = TARGET_SIZE - len(curated)
per_cat = max(MIN_PER_PRIMARY, remaining // max(len(TIER_2), 1))

for cat, items in TIER_2.items():
    added = 0
    for p in items:
        if added >= per_cat:
            break
        if add_prompt(p):
            added += 1

print(f'After category fill: {len(curated)}')

# Step 3: Fill remaining slots from largest categories
if len(curated) < TARGET_SIZE:
    all_remaining = []
    for cat, items in TIER_2.items():
        for p in items:
            if p['text'][:100].lower().strip() not in seen_prefixes:
                all_remaining.append(p)
    all_remaining.sort(key=source_score, reverse=True)
    for p in all_remaining:
        if len(curated) >= TARGET_SIZE:
            break
        add_prompt(p)

print(f'Final curated set: {len(curated)}')


## 4. Curated set statistics

The stats below verify that the curated set meets our targets:
- All graded prompts included (for calibration)
- All 5 primary categories represented (for coverage)
- Difficulty levels balanced (for comprehensive evaluation)
- Total size manageable within a Kaggle GPU session


In [ ]:
curated_cats = Counter(get_primary_category(p) for p in curated)
curated_diff = Counter(p.get('difficulty', 'unknown') for p in curated)
curated_graded = sum(1 for p in curated if p.get('graded_responses'))

print(f'Curated prompts:     {len(curated):,}')
print(f'Graded (with refs):  {curated_graded}')
print(f'Ungraded:            {len(curated) - curated_graded}')
print(f'\nBy primary category:')
for cat, n in curated_cats.most_common():
    pct = n / len(curated) * 100
    print(f'  {cat:<45} {n:>5} ({pct:>5.1f}%)')
print(f'\nBy difficulty:')
for diff, n in curated_diff.most_common():
    print(f'  {diff:<20} {n:>5}')

# Show a few example prompts per category
print(f'\n--- Sample prompts ---')
shown = set()
for p in curated:
    cat = get_primary_category(p)
    if cat in shown:
        continue
    shown.add(cat)
    print(f'\n[{cat}] {p["id"]}')
    print(f'  {p["text"][:150]}...')


## 5. Save curated set

Save the curated subset to `curated_prompts.jsonl` in JSONL format.
This file feeds into the next two stages of the data pipeline.


In [ ]:
import json

output_path = 'curated_prompts.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for p in curated:
        f.write(json.dumps(p, ensure_ascii=False, default=str) + '\n')

print(f'Saved {len(curated):,} curated prompts to {output_path}')
print(f'This file feeds into Notebook 00 (Gemma Exploration).')
print(f'\nTo use in Notebook 00:')
print(f'  prompts = [json.loads(line) for line in open("curated_prompts.jsonl")]')


## Summary and next steps

### What this produces

A balanced, prioritized subset of the full 74,567-prompt corpus:
- All 204 graded prompts (with 5-level reference responses for calibration)
- Coverage across all 5 primary rubric categories
- Difficulty balance (basic / medium / hard)
- Near-duplicates removed
- Source quality prioritized (manual > legacy > generated > untested)

### Connection to the pipeline

- **Next (NB 00b):** The Prompt Remixer takes this curated set and
  generates adversarial variations (academic framing, role-play,
  corporate wrapping, urgency pressure, corridor swaps)
- **Then (NB 00):** Gemma Exploration runs the combined set through
  stock Gemma 4 E4B and scores every response using the weighted rubric

### Why the curation matters

Running 74K prompts through Gemma would take 103 hours. The curated
subset captures the same information in ~17 minutes on T4 GPU. The
prioritization ensures we evaluate what matters most: graded prompts
for calibration, category coverage for completeness, and difficulty
balance for robustness.

This is the foundation that every subsequent evaluation builds on.
